# GFTM default vs m11_new against GS2: NSTX MTM hypercube

Do GFTM at its default settings (`gftm_fw174`: `FIND_WIDTH=F`, `WIDTH=1.74`) and the `m11_new` settings reproduce the GS2 growth rate on the 1000-point NSTX MTM Latin hypercube (`beta_q_shat_ky_n1000`)? Both arms are compared to GS2 on the same cases, so the two panels share one population.

Selection: GS2 cases with $\gamma > 10^{-3}$, and GFTM's dominant mode (`argmax` of `growth_rate` over `mode`). Only cases where **both** arms return $\gamma > 0$ are scored (the paired population). Metrics use finite, strictly positive pairs: RMSE and bias raw in $c_s/a$, Pearson $r$ on $\log_{10}\gamma$. Cases are matched by `sample_name`, never by position.

Each scored case is also **classified by mode type**, with pyrokinetics' own `FieldLine.compute_linear_tearing_parameter`. It returns $T = \left|\int A_\parallel \sqrt{g_{\theta\theta}}\,\mathrm{d}\theta\right| \big/ \int \left|A_\parallel \sqrt{g_{\theta\theta}}\right|\,\mathrm{d}\theta$: $T \to 1$ for an **even** (tearing-parity) $A_\parallel$, $T \to 0$ for an odd one. `NSTX_MTM_LHC` is run with a parity-mixed initial condition and no parity constraint, so the GS2 mode type here is measured rather than imposed. The classification is of **GS2**'s mode only — see the interpretation cell for why the diagnostic cannot be applied to the GFTM arms.

## Imports and settings

Set `GK_DATA_ROOT` in `local.env` to the directory containing `GS2/` and `GFTM/`. Each arm is a per-code entry of scan information under the same project and case.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
from pyrokinetics import Pyro, PyroHypercube
from pyrokinetics.diagnostics.field_line import FieldLine

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "mtm_default_vs_m11"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
run_template = "Runs"
project = "LATIN_HYPERCUBE"
case = "NSTX_MTM"
scan = "beta_q_shat_ky_n1000"
metadata_file = "pyroscan.json"
output_file = "cube.nc"

reference = ("GS2", f"{scan}/pyro_cube_avg")
arms = {  # panel title -> (code, scan information)
    "GFTM default": ("GFTM", f"{scan}/gftm_fw174/pyro_cube"),
    "GFTM m11_new": ("GFTM", f"{scan}/m11_new/pyro_cube"),
}
gs2_unstable = 1e-3  # c_s/a; GS2 cases at or below this are not scored
tearing_threshold = 0.5  # T above this is called tearing parity (even A_par)

## Load data

Pyrokinetics loads each hypercube with its base input and GK output. Rerun only when inputs change.

In [ ]:
def load(code, scan_information):
    directory = data_root / code / run_template / project / case / scan_information
    cube = PyroHypercube(pyroscan_json=directory / metadata_file, load_base_pyro=True)
    cube.from_netcdf(directory / output_file)
    return cube

gs2_cube = load(*reference)
gs2 = gs2_cube.gk_output.data
data = {title: load(*spec).gk_output.data for title, spec in arms.items()}
print("GS2", dict(gs2.sizes))
for title, ds in data.items():
    print(title, dict(ds.sizes))

## Calculate and select

Arms are reindexed onto the GS2 `sample_name` order (a missing case raises). The dominant mode is the `argmax` of `growth_rate` over `mode`; rows with no finite mode become NaN and drop out of the metrics. The paired mask keeps GS2-unstable cases where every arm has $\gamma > 0$. The count of selected modes that are not index 0 is printed: argmax and index 0 coincide only if it is 0.

In [ ]:
names = [str(n) for n in gs2.sample_name.values]
gamma_gs2 = np.asarray(gs2.growth_rate.values, float)

gamma, nonzero = {}, {}
for title, ds in data.items():
    ds = ds.isel(sample=[list(map(str, ds.sample_name.values)).index(n) for n in names])
    g = np.asarray(ds.growth_rate.transpose("sample", "mode").values, float)  # plain c_s/a numbers
    g = np.where(np.isfinite(g), g, -np.inf)
    k = g.argmax(axis=1)
    gamma[title] = np.where(np.isfinite(g.max(axis=1)), g.max(axis=1), np.nan)
    nonzero[title] = k

paired = (gamma_gs2 > gs2_unstable) & np.all([g > 0 for g in gamma.values()], axis=0)
print(f"GS2-unstable: {(gamma_gs2 > gs2_unstable).sum()}, paired: {paired.sum()}")
print({t: int((nonzero[t][paired] != 0).sum()) for t in nonzero}, "= selected mode not index 0")

stats = {}
for title, g in gamma.items():
    x, y = gamma_gs2[paired], g[paired]
    ok = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x, y = x[ok], y[ok]
    stats[title] = dict(n=len(x), rmse=np.sqrt(np.mean((y - x) ** 2)), bias=np.mean(y - x),
                        r=np.corrcoef(np.log10(x), np.log10(y))[0, 1])
    print(title, stats[title])

## Classify the GS2 mode

`FieldLine.compute_linear_tearing_parameter` takes a `Pyro`, and reads `gk_output["apar"]` at
`(kx=0, ky=0, time=-1)` together with the metric term $g_{\theta\theta}$ of that case's geometry. The
hypercube's stacked output is not a `Pyro`, and the per-sample `Pyro`s that `build_pyro_dict()` creates
carry the **base** geometry with the sampled values held unapplied in `run_parameters` — they are for
writing decks, not for reading one back. So each scored case is read from **its own run directory**,
whose deck `Pyro` parses into the geometry that case actually ran (`gs2_cube.base_directory` and
`gs2_cube.file_name` are the cube's own record of where those runs are). Nothing here computes the
parity integral; the diagnostic does.

GS2 has one mode per case, so no mode selection is involved and the classified mode is necessarily the
one whose growth rate is plotted. Note the diagnostic reads the **last time point**, while the cube's
`growth_rate` is averaged over the tail of the trace; for a converged linear eigenmode the structure at
the final step is the eigenmode, but the two are not the same reduction. `nperiod = 9` here, so parity is
measured over the full $\pm 17\pi$ ballooning domain the runs used.

This cell reads ~800 GS2 outputs one at a time and takes roughly 20 minutes.

In [ ]:
runs = Path(gs2_cube.base_directory)
tearing = np.full(len(names), np.nan)
for i in np.flatnonzero(paired):
    pyro = Pyro(gk_file=runs / names[i] / gs2_cube.file_name)  # that case's own deck and geometry
    pyro.load_gk_output(load_fluxes=False, load_moments=False)
    tearing[i] = FieldLine(pyro).compute_linear_tearing_parameter()

assert np.isfinite(tearing[paired]).all(), "diagnostic returned a non-finite T"
is_tearing = tearing > tearing_threshold
print(f"GS2 tearing parity: {is_tearing[paired].mean():.3f} of n={paired.sum()}"
      f" at T > {tearing_threshold}; T spans [{np.nanmin(tearing):.3f}, {np.nanmax(tearing):.3f}]")
print("fraction sensitivity", {t: round(float((tearing[paired] > t).mean()), 3) for t in (0.3, 0.4, 0.5, 0.6, 0.7)})
for title, g in gamma.items():
    for label, mask in (("tearing", is_tearing), ("other  ", ~is_tearing)):
        m = paired & mask
        print(f"  {title} {label}: n={m.sum():3d} RMSE {np.sqrt(np.mean((g[m] - gamma_gs2[m]) ** 2)):.4f}"
              f" median ratio {np.median(g[m] / gamma_gs2[m]):.3f}")

## Plot

One log-log parity panel per arm, GS2 on the x axis with $y=x$ dashed, on shared axes limits. Markers
split the shared population by the GS2 mode type: tearing parity (MTM-like) against everything else.

In [ ]:
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(10, 5))
lim = (1e-3, 2 * max(np.nanmax(gamma_gs2[paired]), *(np.nanmax(g[paired]) for g in gamma.values())))
classes = (("GS2 tearing parity", is_tearing, "o", "C0"), ("other parity", ~is_tearing, "x", "C1"))
for ax, (title, g) in zip(axes, gamma.items()):
    s = stats[title]
    for label, mask, marker, color in classes:
        m = paired & mask
        ax.scatter(gamma_gs2[m], g[m], s=8, alpha=0.5, marker=marker, color=color,
                   label=f"{label} (n={m.sum()})")
    ax.plot(lim, lim, "k--", marker="")
    ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim, aspect="equal",
           xlabel=r"GS2 $\gamma\,a/c_s$", title=f"{title} (n={s['n']})")
    ax.text(0.04, 0.96, f"RMSE {s['rmse']:.4f}\nbias {s['bias']:+.4f}\nr {s['r']:.4f}",
            transform=ax.transAxes, va="top")
axes[0].set_ylabel(r"GFTM $\gamma\,a/c_s$")
axes[0].legend(loc="lower right")
plt.show()

## Save

Replaces the same filename.

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / f"{scan}_paired.png")

## Interpretation

Compare the printed metrics with the reference table `results/nstx_mtm_lhc_default_vs_m11/csv/summary_default_vs_m11.csv` (n1000, paired): default RMSE 0.2790 / bias +0.0877 / r 0.5754, m11_new 0.3093 / +0.1272 / 0.5945, n = 833.

Both arms are compared on the same cases, so differences between panels are not a population effect. m11_new correlates slightly better (higher $r$) but overpredicts more (larger positive bias and RMSE).

**The GS2 tearing fraction, against the earlier measurement.** `Fusion_PhD-suy.21` measured 0.807 (n300) / 0.808 (n1000) for GS2 on this same data with a different detector. This diagnostic gives **0.701** on the n1000 paired population, and no threshold recovers 0.81 (0.736 at $T>0.3$, 0.592 at $T>0.7$), so the offset is in the statistic and not in the cut. The direction is the expected one. $T$ compares the **magnitude of the integral** against the **integral of the magnitude**, so any variation of $A_\parallel$'s complex phase across an even envelope reduces it; a parity index instead projects onto the even and odd subspaces and asks only which is larger, which a predominantly-even mode passes cleanly however its phase behaves. $T$ is therefore strictly the more demanding measure of the two, and the full $\pm 17\pi$ window used here compounds that against suy.21's $\pm 9\pi$ by including tails where the envelope is small and noisy. Two further differences act the same way: that detector required tearing parity in **both** $A_\parallel$ and $\phi$ where this uses $A_\parallel$ alone, and it read a tail-averaged mode where this reads the last time point. 0.70 against 0.81 is the same population scored more strictly, not a different classification — it corroborates the earlier number rather than contradicting it.

**What the split adds to the growth-rate comparison.** The two arms fail differently on the two populations, and the difference is larger than either arm's overall bias suggests. On the GS2 tearing-parity cases (n = 584) the default **under**-predicts, median $\gamma$ ratio 0.80, while `m11_new` over-predicts by 1.75; on the remaining 249 cases both sit near unity (1.08 and 1.14). So `m11_new`'s well-known ~55% over-prediction on this database is not spread across the population — it is concentrated almost entirely on the modes GS2 says are MTMs, which is the population it was tuned for. Its RMSE is also worse than the default's on **both** classes (0.288 vs 0.273 tearing, 0.354 vs 0.293 other).

**The GFTM arms are not classified, and this is a pyrokinetics gap, not a choice.** `compute_linear_tearing_parameter` reads `gk_output["apar"]` over `theta`. For GS2 that variable exists. For GFTM (and TGLF) it does not: the reader's `_get_fields` returns field **amplitudes** per `(ky, mode)`, with no `theta` axis, and the theta-resolved information is carried instead in `eigenfunctions(field, theta, mode)`. A GFTM `gk_output` therefore has no `apar` at all and the call raises `KeyError`. Since $T$ is invariant under multiplication of $A_\parallel$ by any complex constant, and `eigenfunctions` is exactly the field divided by such a constant, the diagnostic would give the same answer on `eigenfunctions` — but making it do so means changing pyrokinetics, which is not this notebook's call. Reproducing the tearing-parity fractions suy.21 measured for the GFTM arms (0.553 / 0.539 default, 0.734 / 0.714 m11_new) is blocked on that change, and with it the per-arm mode-type comparison that was the sharper half of that result. Nothing is reimplemented here in its place.